# Türkiye yatırım notebook'u — daha okunur sürüm

Bu notebook son 6 ay için her ay maaştan yatırım yapıldığında üç stratejinin nasıl davrandığını test eder.

Temel farklar:
- veri kapsamı zayıf olan varlıklar otomatik elenir
- altın artık tek günlük zayıf bir Yahoo sembolüne bağlı değildir; **ons altın × USD/TRY** ile yaklaşık **TL bazlı altın** serisi üretilir
- ay başı tarihler ilk işlem gününe eşlenir
- HRP ve Black-Litterman yetersiz veri durumunda güvenli fallback kullanır
- çıktıların dili ve tablolar daha anlaşılır tutulur


In [ ]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai

In [ ]:

import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier


## Nasıl kullanılır?

Bu notebook bir yatırım tavsiyesi üretmez.  
Amaç, **aynı veri üzerinde** birkaç portföy yaklaşımını karşılaştırmaktır.

Bu notebook şu soruya cevap arar:

> Son 6 ay boyunca her ay aynı miktarda para yatırsaydım, farklı portföy kuralları nasıl davranırdı?

Okurken özellikle şunlara bak:
- **Veri kaliteli mi?**
- **Varlık evreni mantıklı mı?**
- **Strateji çok tek varlığa mı yığılıyor?**
- **Getiri farkı gerçekten anlamlı mı, yoksa küçük mü?**


In [ ]:

CONFIG = {
    "stock_tickers": ["THYAO.IS", "AKBNK.IS", "EREGL.IS"],
    "fx_ticker": "USDTRY=X",
    "gold_usd_ticker": "GC=F",
    "asset_labels": {
        "THYAO.IS": "THYAO",
        "AKBNK.IS": "AKBNK",
        "EREGL.IS": "EREGL",
        "USDTRY=X": "USDTRY",
        "GOLD_TRY": "ALTIN_TRY",
    },
    "salary_try": 25000,
    "target_test_months": 6,
    "target_lookback_days": 252,
    "download_period": "10y",
    "max_weight": 0.70,
    "gold_label": "ALTIN_TRY",
    "gold_min": 0.10,
    "gold_max": 0.55,
    "min_history_days": 180,
    "use_llm_views": False,
    "llm_model": "gpt-4.1-mini",
}
CONFIG


In [ ]:

def try_fmt(x):
    if pd.isna(x):
        return "-"
    return f"₺{x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct_fmt(x):
    if pd.isna(x):
        return "-"
    return f"{x*100:.2f}%"

def style(fig, title):
    fig.update_layout(
        template="plotly_white",
        title=title,
        hovermode="x unified",
        legend_title_text=""
    )
    return fig

def extract_close_series(df):
    if df is None or len(df) == 0:
        return None
    if isinstance(df, pd.Series):
        s = df.dropna()
        return s if len(s) else None

    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        if "Close" in level0:
            close = df.xs("Close", axis=1, level=0)
        else:
            close = df.iloc[:, :1]
    else:
        close = df[["Close"]] if "Close" in df.columns else df.iloc[:, :1]

    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]

    s = pd.Series(close).dropna()
    return s if len(s) else None

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return w / w.sum()
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return w / w.sum()
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return w / w.sum()

def clean_returns(price_window):
    rets = price_window.pct_change()
    rets = rets.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    if rets.empty:
        return rets
    valid_cols = []
    for c in rets.columns:
        s = rets[c]
        if np.isfinite(s).all() and s.std() > 1e-10:
            valid_cols.append(c)
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def views_from_prices(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return {c: 0.0 for c in price_window.columns}
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    score = score.reindex(price_window.columns).fillna(0.0)
    return score.clip(-0.15, 0.15).to_dict()

def llm_views_from_prices(price_window, model="gpt-4.1-mini"):
    labels = list(price_window.columns)
    rule_views = views_from_prices(price_window)

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return rule_views, "rules_no_key"

    try:
        from openai import OpenAI

        recent = clean_returns(price_window.tail(60))
        if recent.empty:
            return rule_views, "rules_insufficient_data"

        ann = (recent.mean() * 252).round(4).to_dict()
        vol = (recent.std() * np.sqrt(252)).round(4).to_dict()
        mom20 = ((price_window.iloc[-1] / price_window.iloc[-20]) - 1).round(4).to_dict() if len(price_window) >= 20 else {}
        mom60 = ((price_window.iloc[-1] / price_window.iloc[-60]) - 1).round(4).to_dict() if len(price_window) >= 60 else {}

        prompt = {
            "task": "Return annual expected return views for Black-Litterman.",
            "rules": [
                "Return valid JSON only.",
                "Keys must exactly match the asset labels provided.",
                "Values must be floats between -0.15 and 0.15.",
                "Do not add any explanation."
            ],
            "assets": labels,
            "features": {
                "annualized_recent_return": ann,
                "annualized_volatility": vol,
                "momentum_20d": mom20,
                "momentum_60d": mom60
            },
            "output_example": {k: 0.01 for k in labels}
        }

        client = OpenAI(api_key=api_key)
        resp = client.responses.create(
            model=model,
            input=[
                {"role": "system", "content": "You are a quantitative portfolio assistant. Output strict JSON only."},
                {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)}
            ],
            temperature=0
        )

        text = getattr(resp, "output_text", None)
        if text is None:
            text = str(resp)

        parsed = json.loads(text)
        views = {k: float(parsed.get(k, 0.0)) for k in labels}
        views = {k: float(min(max(v, -0.15), 0.15)) for k, v in views.items()}
        return views, "llm"

    except Exception as e:
        print("LLM failed, inverse-vol değil kurallı view fallback kullanıldı:", str(e))
        return rule_views, "rules_fallback"


## 1) Veri kapsamı ve kullanılan varlıklar

Bu bölüm en kritik kontrollerden biridir.

İlk tablo:
- her veri kaynağının kaç gözlemi olduğunu,
- başlangıç ve bitiş tarihini,
- veri alınıp alınamadığını gösterir.

İkinci tablo:
- hangi varlıkların gerçekten testte kullanıldığını gösterir.

Burayı şöyle yorumla:
- Eğer bir varlığın tarihi çok kısa ise test bozulur.
- Eğer ortak tarihçe çok kısa ise sonuçlara güvenilmez.
- Eğer burada her şey uzunsa, test artık teknik olarak anlamlıdır.


In [ ]:

raw_rows = []
series = []

tickers_to_download = CONFIG["stock_tickers"] + [CONFIG["fx_ticker"], CONFIG["gold_usd_ticker"]]

downloaded = {}
for t in tickers_to_download:
    df = yf.download(t, period=CONFIG["download_period"], auto_adjust=True, progress=False)
    s = extract_close_series(df)
    downloaded[t] = s
    if s is None:
        raw_rows.append({"source": t, "asset": CONFIG["asset_labels"].get(t, t), "n": 0, "status": "empty"})
    else:
        raw_rows.append({
            "source": t,
            "asset": CONFIG["asset_labels"].get(t, t),
            "n": len(s),
            "start": str(s.index.min().date()),
            "end": str(s.index.max().date()),
            "status": "ok"
        })

# build synthetic gold in TRY
gold_usd = downloaded.get(CONFIG["gold_usd_ticker"])
usdtry = downloaded.get(CONFIG["fx_ticker"])

if gold_usd is not None and usdtry is not None:
    gold_try = (gold_usd * usdtry).dropna()
    gold_try.name = CONFIG["asset_labels"]["GOLD_TRY"]
    raw_rows.append({
        "source": f"{CONFIG['gold_usd_ticker']} * {CONFIG['fx_ticker']}",
        "asset": CONFIG["asset_labels"]["GOLD_TRY"],
        "n": len(gold_try),
        "start": str(gold_try.index.min().date()) if len(gold_try) else None,
        "end": str(gold_try.index.max().date()) if len(gold_try) else None,
        "status": "ok" if len(gold_try) else "empty"
    })
else:
    gold_try = None
    raw_rows.append({
        "source": f"{CONFIG['gold_usd_ticker']} * {CONFIG['fx_ticker']}",
        "asset": CONFIG["asset_labels"]["GOLD_TRY"],
        "n": 0,
        "status": "empty"
    })

coverage = pd.DataFrame(raw_rows)
display(coverage)

# keep only assets with enough history
for t in CONFIG["stock_tickers"] + [CONFIG["fx_ticker"]]:
    s = downloaded.get(t)
    if s is not None and len(s) >= CONFIG["min_history_days"]:
        s = s.copy()
        s.name = CONFIG["asset_labels"][t]
        series.append(s)

if gold_try is not None and len(gold_try) >= CONFIG["min_history_days"]:
    series.append(gold_try)

eligible_assets = pd.DataFrame({
    "asset": [s.name for s in series],
    "n": [len(s) for s in series],
    "eligible": True
})
display(eligible_assets)

if len(series) < 3:
    raise ValueError("Yeterli tarihçesi olan en az 3 varlık yok. Varlık evrenini değiştir.")

prices = pd.concat(series, axis=1).sort_index().ffill()

firsts = prices.apply(lambda s: s.first_valid_index())
common_start = firsts.max()
prices = prices.loc[common_start:].dropna()

monthly_points = prices.resample("MS").first().dropna()
available_months = len(monthly_points)

if available_months < CONFIG["target_test_months"] + 1:
    raise ValueError(f"6 aylık test için yeterli ortak tarihçe yok. Kullanılabilir aylık nokta sayısı: {available_months}")

lookback = min(CONFIG["target_lookback_days"], max(60, len(prices) - 40))
months = CONFIG["target_test_months"]

print("Kullanılan varlıklar:", list(prices.columns))
print("Toplam gün:", len(prices), "lookback:", lookback, "test_months:", months)
display(prices.tail())


## 2) Piyasa davranışı: normalize fiyatlar ve korelasyon

İlk grafik: **Normalize fiyatlar**
- Bütün varlıklar 100'den başlatılır.
- Amaç, hangisinin daha iyi gittiğini görmek.
- Fiyat seviyesi değil, **performans eğrisi** önemlidir.

İkinci grafik: **Korelasyon**
- 1'e yakınsa birlikte hareket ederler.
- 0 civarıysa ilişki zayıftır.
- Negatifse biri düşerken diğeri dengeleyebilir.

Buradaki amaç:
- Portföy gerçekten çeşitleniyor mu?
- Yoksa farklı isimlerle aynı şeye mi yatırım yapıyoruz?


In [ ]:

fig = px.line(prices.tail(504) / prices.tail(504).iloc[0] * 100, x=prices.tail(504).index, y=prices.columns)
style(fig, "Son ~2 yıl normalize fiyatlar").show()

fig = px.imshow(
    prices.pct_change().dropna().corr(),
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
style(fig, "Korelasyon").show()


## 3) Stratejiler ne yapıyor?

Bu notebook üç yaklaşımı karşılaştırır:

### Equal
Her varlığa eşit ağırlık verir.  
En basit ve en dürüst baseline budur.

### HRP
Riski dağıtmaya çalışır.  
Pratikte bazen daha sakin görünen varlıklara fazla yüklenebilir.

### BL
Black-Litterman yaklaşımıdır.  
Burada görüşler:
- ya kural tabanlıdır,
- ya istenirse LLM ile üretilebilir.

Önemli not:
Bu sürümde HRP ve BL, veri yetersizse güvenli fallback kullanır.  
Yani hata vermek yerine daha basit bir ağırlıklandırmaya döner.


In [ ]:

def ew_weights(cols):
    return pd.Series(1 / len(cols), index=cols)

def fallback_inverse_vol(price_window):
    rets = clean_returns(price_window)
    w = pd.Series(0.0, index=price_window.columns)

    if rets.empty or rets.shape[1] == 0:
        return ew_weights(price_window.columns)

    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()

    if iv.empty:
        return ew_weights(price_window.columns)

    iv = iv / iv.sum()
    w.loc[iv.index] = iv
    return w / w.sum()

def hrp_weights(price_window):
    rets = clean_returns(price_window)

    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return fallback_inverse_vol(price_window)

    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)

        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub

        if w.sum() <= 0:
            return fallback_inverse_vol(price_window)

        return w / w.sum()

    except Exception as e:
        print("HRP failed, inverse-vol kullanıldı:", str(e))
        return fallback_inverse_vol(price_window)

def bl_weights(price_window):
    rets = clean_returns(price_window)

    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = fallback_inverse_vol(price_window)
        diag = pd.DataFrame({
            "asset": list(price_window.columns),
            "view": [views_from_prices(price_window).get(c, 0.0) for c in price_window.columns],
            "view_source": "rules_insufficient_data"
        })
        return w, diag

    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()

        if CONFIG["use_llm_views"]:
            views, view_source = llm_views_from_prices(price_window, model=CONFIG["llm_model"])
        else:
            views, view_source = views_from_prices(price_window), "rules"

        bl = BlackLittermanModel(S, pi="equal", absolute_views=views, omega="default")
        post = bl.bl_returns()

        ef = EfficientFrontier(post, S, weight_bounds=(0, CONFIG["max_weight"]))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)

        if w.sum() <= 0:
            w = fallback_inverse_vol(price_window)
            view_source = f"{view_source}_fallback"
        else:
            w = w / w.sum()

        w = band_gold(w, CONFIG["gold_label"], CONFIG["gold_min"], CONFIG["gold_max"])

        diag = pd.DataFrame({
            "asset": list(views.keys()),
            "view": list(views.values()),
            "view_source": view_source
        })
        return w, diag

    except Exception as e:
        print("BL failed, inverse-vol kullanıldı:", str(e))
        w = fallback_inverse_vol(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({
            "asset": list(fallback_views.keys()),
            "view": list(fallback_views.values()),
            "view_source": "rules_bl_fallback"
        })
        return w, diag

def run_strategy(name):
    monthly = prices.resample("MS").first().dropna()
    month_starts = monthly.index[-months:]

    rebalance_dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            rebalance_dates.append(prices.index[idx])

    rebalance_dates = pd.Index(pd.unique(pd.Index(rebalance_dates)))

    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []

    for i, dt in enumerate(rebalance_dates):
        cash += CONFIG["salary_try"]
        window = prices.loc[:dt].tail(lookback)

        if name == "Equal":
            w = ew_weights(prices.columns)
            diag = None
        elif name == "HRP":
            w = hrp_weights(window)
            diag = None
        else:
            w, diag = bl_weights(window)

        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (w * total) / current
        cash = 0.0

        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)

        for t, v in vals.items():
            hist.append({"date": t, "strategy": name, "portfolio_value": float(v)})

        row = {"date": dt, "strategy": name, "capital_after_contribution": total}
        for c, x in w.items():
            row[f"weight_{c}"] = float(x)
        rebs.append(row)

        if diag is not None:
            tmp = diag.copy()
            tmp["date"] = dt
            tmp["strategy"] = name
            diags.append(tmp)

    hist_df = pd.DataFrame(hist).drop_duplicates(["date", "strategy"])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

parts_h, parts_w, parts_d = [], [], []
for s in ["Equal", "HRP", "BL"]:
    h, w, d = run_strategy(s)
    parts_h.append(h)
    parts_w.append(w)
    if not d.empty:
        parts_d.append(d)

hist_df = pd.concat(parts_h, ignore_index=True)
weights_df = pd.concat(parts_w, ignore_index=True)
diag_df = pd.concat(parts_d, ignore_index=True) if parts_d else pd.DataFrame()

display(hist_df.tail())


## 4) Özet tablo nasıl okunur?

### Contributed
Toplam yatırdığın para.

### Ending Value
Test sonunda portföyün ulaştığı değer.

### Net Profit
Bitiş değeri - yatırdığın toplam para.

### TWR
Şu an pratik karşılaştırma metriği gibi kullanılıyor.  
Gerçek kurumsal anlamda tam TWR değildir; daha sonra iyileştirilebilir.

### Ann Vol
Dalgalanma seviyesi.  
Yüksekse daha oynaktır.

### Sharpe
Kabaca getiri / risk verimliliği gibi okunur.  
Daha yüksek olması genelde daha iyidir.

### MaxDD
En kötü düşüş.  
Örneğin -10% ise portföy zirvesinden en fazla %10 aşağı gelmiştir.

Bu tabloyu yorumlarken:
- sadece en yüksek getiriyi değil,
- **getiri + düşüş + ağırlık yapısını birlikte** oku.


In [ ]:

rows = []
contributed = CONFIG["salary_try"] * months

for s in ["Equal", "HRP", "BL"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1

    rows.append({
        "Strategy": s,
        "Contributed": try_fmt(contributed),
        "Ending Value": try_fmt(sub["portfolio_value"].iloc[-1]),
        "Net Profit": try_fmt(sub["portfolio_value"].iloc[-1] - contributed),
        "TWR": pct_fmt(sub["portfolio_value"].iloc[-1] / contributed - 1),
        "Ann Vol": pct_fmt(sub["ret"].std() * np.sqrt(252)),
        "Sharpe": f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
        "MaxDD": pct_fmt(dd.min()),
    })

summary = pd.DataFrame(rows)
display(summary)

fig = px.line(hist_df, x="date", y="portfolio_value", color="strategy")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
style(fig, "Portföy değeri").show()


## 5) Son ağırlıklar ve yeniden dengeleme tablosu

İlk tablo:
- stratejinin en son noktada portföyü nasıl böldüğünü gösterir.

Bunu şöyle yorumla:
- tek varlık çok baskın mı?
- altın / döviz / hisse dengesi mantıklı mı?
- strateji yatırımcı gibi mi davranıyor, yoksa tek temaya mı yığılıyor?

İkinci tablo:
- her ay başında sistemin hangi ağırlıkları seçtiğini gösterir.

Son tablo:
- Black-Litterman tarafında kullanılan görüşleri gösterir.
- Eğer birçok görüş sürekli tavana vuruyorsa, view sistemi fazla kaba olabilir.

Pratik yorum kuralı:
- **çok küçük getiri farkları** varsa,
- ama biri çok daha konsantre ise,
o stratejiye temkinli yaklaş.


In [ ]:

last_weights = weights_df.sort_values("date").groupby("strategy").tail(1).copy()
cols = [c for c in last_weights.columns if c.startswith("weight_")]
out = last_weights[["strategy"] + cols].copy()
out.columns = ["Strategy"] + [c.replace("weight_", "") for c in cols]
for c in out.columns[1:]:
    out[c] = out[c].map(pct_fmt)
display(out.reset_index(drop=True))

rebalance_preview = weights_df.tail(9).copy()
display(rebalance_preview)

if len(diag_df):
    diag_show = diag_df.copy()
    diag_show["view"] = diag_show["view"].map(pct_fmt)
    display(diag_show.tail(15))
